# MMS-300M step-time benchmark

Times training steps on synthetic audio under several settings.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

SESSION_START = time.time()


def find_repo():
    for depth in range(1, 5):
        for hit in Path("/kaggle/input").glob("*/" * depth + "src/train.py"):
            return hit.parent.parent
    raise FileNotFoundError("Code not found: attach the waxal-asr-code dataset (Add Input).")


REPO = find_repo()
PY = "/tmp/waxal-venv/bin/python"
os.environ.update(HF_HOME="/tmp/hf_cache", HF_DATASETS_DISABLE_PROGRESS_BARS="1", WANDB_DISABLED="true",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
                  # One GPU: Trainer's DataParallel over both T4s made MMS-300M steps ~6.5x
                  # slower (kaggle/bench_mms.py). Accumulation doubles to keep batch 32.
                  CUDA_VISIBLE_DEVICES="0")
TOKEN_FILES = sorted(Path("/kaggle/input").glob("**/hf_token.txt"))
if TOKEN_FILES:
    os.environ["HF_TOKEN"] = TOKEN_FILES[0].read_text().strip()
else:
    print("No hf_token.txt input (waxal-hf-token dataset); checkpoints will not survive this session.")


LOG_DIR = Path("/kaggle/working/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run(*command, check=True):
    """Run a command from the repo root, streaming its output to the notebook and to
    /kaggle/working/logs (kept in the job output; the API's notebook log can be empty)."""
    name = "_".join(Path(str(part)).stem for part in command[1:4])
    log_path = LOG_DIR / f"{time.strftime('%H%M%S')}_{name}.log"
    process = subprocess.Popen(list(command), cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    with open(log_path, "w", encoding="utf-8") as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
    if process.wait() and check:
        raise RuntimeError(f"{' '.join(map(str, command))} exited with {process.returncode}")
    return process.returncode


print("Code:", REPO)

In [ ]:
run("bash", f"{REPO}/kaggle/setup_env.sh")

In [ ]:
run(PY, f"{REPO}/kaggle/bench_mms.py")